# Exploring Belgium from Space with JupyterGIS

[JupyterGIS](https://jupytergis.readthedocs.io) is a JupyterLab extension that brings an interactive GIS environment directly into your notebook. It lets you:

- Visualise raster and vector layers on an interactive map
- Stack, reorder, and adjust the opacity of multiple layers from the layer panel
- Load data from a wide range of formats, like **GeoJSON, GeoTIFF, Shapefile, GeoPackage, WMS/WMTS** and more
- Add story segments to tell interactive stories.

In this notebook we use the **Python API** to build a map programmatically. We will:

1. Create an interactive map centred on Belgium
2. Discover which Terrascope radar collections can be visualised through WMTS
3. Add a **Sentinel-1 Sigma0** layer from Terrascope
4. Add an **OPERA S1 RTC** layer for comparison


## 1. Create the interactive map

We start by creating a `GISDocument`, the main JupyterGIS object. It renders as an interactive map widget inline in the notebook. The parameters set the initial view over Belgium.

Once it appears you can already:
- **Pan and zoom** with the mouse
- Scroll by default available layers (book icon, top left). 
- Add custom **Layers** (plus icon, top-left)
- Add **markers** and **story segments**

In [9]:
from jupytergis import GISDocument

doc = GISDocument(latitude=50.5, longitude=4.5, zoom=7)
doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap"
)
doc

## 2. Discovering radar datasets on Terrascope

[Terrascope](https://terrascope.be) exposes its Earth Observation archive through a [STAC](https://stacspec.org) (SpatioTemporal Asset Catalog) API. Each **collection** describes the metadata of a dataset.

Some collections support the **STAC Renders extension**, which are used by TiTiler to render the data of the collection through the Terrascope WMTS endpoint. This is what we'll use to add layers to JupyterGIS without any manual processing.

The cell below queries the Terrascope STAC, builds a lookup of render keys for all renderable collections, and then prints only those tagged as **radar** datasets,the ones relevant to our Sentinel-1 exploration.

In [10]:
import requests

response = requests.get("https://stac.terrascope.be/collections")
collections = response.json()["collections"]

# Build a render-key lookup for ALL renderable collections
wmts_name = {}
for c in collections:
    if "renders" in c:
        for render in c["renders"]:
            wmts_name[c["id"]] = c["id"] + "_" + render

# Print only the radar collections so we can see what SAR data is available
radar_collections = [
    c for c in collections
    if "renders" in c and "sentinel-1" in [kw.lower() for kw in c.get("keywords", [])]
]

print(f"Found {len(radar_collections)} radar collections with WMTS render support:\n")
for c in radar_collections:
    title = c.get("title", c["id"])
    render_key = wmts_name.get(c["id"], "\u2014")
    print(f"  \U0001f4e1 {title}")
    print(f"     id:           {c['id']}")
    print(f"     render layer: {render_key}\n")

Found 13 radar collections with WMTS render support:

  📡 CropSAR-2D Near Real Time Fraction Absorbed Photosynthetically Radiation (FAPAR)
     id:           cropsar2d-fapar-nrt
     render layer: cropsar2d-fapar-nrt_fapar

  📡 ESA WorldCereal products for maize per agro-ecological zone - V1
     id:           esa-worldcereal-maize-10m-2021-v1
     render layer: esa-worldcereal-maize-10m-2021-v1_classification

  📡 ESA WorldCereal products for springcereals per agro-ecological zone - V1
     id:           esa-worldcereal-springcereals-10m-2021-v1
     render layer: esa-worldcereal-springcereals-10m-2021-v1_classification

  📡 ESA WorldCereal products for temporary crops per agro-ecological zone - V1
     id:           esa-worldcereal-temporarycrops-10m-2021-v1
     render layer: esa-worldcereal-temporarycrops-10m-2021-v1_classification

  📡 ESA WorldCover Global Land Cover Map for 2020, at 10 meter resolution in COG format
     id:           esa-worldcover-map-10m-2020-v1
     render l

## 3. Building WMTS tile URLs

JupyterGIS can consume any XYZ-style tile URL. The `{z}`, `{x}`, and `{y}` placeholders are filled in automatically by the map widget as the user pans and zooms, each tile request fetches exactly the pixels needed for the current view.

The helper function below constructs the correct Terrascope WMTS URL for a given render layer key. The optional `time` parameter lets us pin the tiles to a specific acquisition date, slicing into the dataset's time series.

In [11]:
def wmts_url_from_stac(layer: str, time: str = None) -> str:
    """Build a Terrascope WMTS XYZ URL from a render layer key and an optional date."""
    url = (
        "https://wmts.terrascope.be/wmts/"
        f"?layer={layer}"
        "&style=default"
        "&tilematrixset=EPSG:3857"
        "&Service=WMTS&Request=GetTile&Version=1.0.0"
        "&Format=image/png"
        "&TileMatrix={z}&TileCol={x}&TileRow={y}"
    )
    if time:
        url += f"&TIME={time}"
    return url

## 4. Sentinel-1 GRD Sigma0 — raw backscatter over Belgium

The **Terrascope Sentinel-1 GRD Sigma0** collection provides ground-range detected SAR imagery processed to sigma-nought (σ⁰) backscatter. By running the cell below, we add the WTMS layer to the above JupyterGIS document.

In [12]:
doc.add_raster_layer(
    url=wmts_url_from_stac(wmts_name["terrascope-s1-grd-sigma0-v1"], time="2026-03-18"),
    name="S1 Sigma0 (March 2026)",
    opacity=0.8
)

'de2aafb2-e3d8-447e-a678-35395de2f06b'

## 5. OPERA S1 RTC — terrain-corrected backscatter

The **OPERA S1 RTC** (Radiometric Terrain Correction) product is similar to GRD Sigma0 and is derived from the same Sentinel-1 GRD data. It would be interesting to compare the two side by side. We can add it as another WMTS layer using the same process as above.

With both layers now loaded you can **compare them interactively** inside JupyterGIS:

1. Open the **Layers panel** and toggle each layer on/off by clicking the 'eye' icon.
2. Drag the **opacity slider** on the top layer to blend between the two
3. Both layers share the same acquisition date (`2026-03-18`), so any differences are due to the RTC processing alone

In [13]:
doc.add_raster_layer(
    url=wmts_url_from_stac(wmts_name["opera-s1-rtc-v1"], time="2026-03-18"),
    name="OPERA S1 RTC (March 2026)",
    opacity=0.8
)

'1d892594-9ccd-4bbb-a365-7da98bbad7d0'

## 6. Beyond this notebook: more JupyterGIS features

This notebook has only scratched the surface of what JupyterGIS can do. Here are some features worth exploring in the UI:

### 📂 Supported file formats
JupyterGIS can open and display many common geospatial formats:

| Format | Type | Notes |
|---|---|---|
| GeoJSON | Vector | Points, lines, polygons: load local files or remote URLs |
| GeoTIFF / COG | Raster | Local files or streamed from object storage |
| Shapefile, GeoPackage | Vector | Traditional GIS formats |
| WMS / WMTS | Raster service | OGC map and tile services (exactly what we used above) |
| XYZ tiles | Raster service | OpenStreetMap-style slippy map tiles |
| `.jGIS` | Project file | JupyterGIS's own format — saves all layers, styles, and camera position for later |

### 🎬 Story maps
**Story maps** let you turn a JupyterGIS map into a guided, step-by-step narrative, think of it as a presentation embedded in the map. Each story point captures:

- A specific **camera position** (centre coordinates + zoom level)
- Which **layers are visible** at that step
- A **text annotation** explaining what the viewer should focus on

## 7. What's next?

Now that you have a working multi-layer S1 map of Belgium, here are some things to try:

- **Change the date** — swap `"2026-03-18"` for another date in the `wmts_url_from_stac` calls and re-run to explore the time series. Sentinel-1 has a ~6-day revisit over Belgium.
- **Explore other radar collections** — the discovery step listed all radar datasets available on Terrascope. Swap in any of those collection IDs to visualise other SAR products.
- **Create a Story Point** — navigate to a view you want to highlight, then use the Story Segments panel to capture it and add an annotation for others to follow. More info can be found here: https://jupytergis.readthedocs.io/en/latest/user_guide/how-tos/story-maps.html